# NHS PROMs — Knee Replacement: Data Preparation

**Source data:** `notebooks/Lalita/UnderstandingData/external/`  
**Output (prepared dataset):** `data/interim/knee-provider.parquet`  
**Years covered:** 2016/17 · 2017/18 · 2018/19  

This notebook cleans and prepares the raw NHS PROMs Knee Replacement Provider-level data  
for downstream modelling. Each step documents *what* is done and *why*, grounding decisions  
in the NHS Digital PROMs Technical Specification and the clinical literature.  

The structure mirrors the Hip Replacement data preparation notebook; differences  
specific to knee data are noted inline.

## 0 · Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)

## 1 · Load raw data

Three annual CSV files (2016/17, 2017/18, 2018/19) are concatenated into a single dataframe.  
File names vary slightly across years (singular vs plural); paths are explicit to avoid silent errors.

In [2]:
# Base directory — external pre-extracted CSVs
EXTERNAL = Path('../../Lalita/UnderstandingData/external')

# Output path for the cleaned, prepared dataset
OUTPUT = Path('../../../data/interim/knee-provider.parquet')

csv_paths = [
    EXTERNAL / 'Knee Replacement Provider 1617.csv',
    EXTERNAL / 'Knee Replacements Provider 1718.csv',   # note: plural in 1718 filename
    EXTERNAL / 'Knee Replacement Provider 1819.csv',
]

# Read and concatenate — low_memory=False prevents dtype inference warnings
# on columns with mixed types (e.g. '*' suppression codes alongside numbers)
frames = [pd.read_csv(p, sep=',', low_memory=False) for p in csv_paths]
df = pd.concat(frames, ignore_index=True)

print(f'Loaded {len(df):,} rows × {df.shape[1]} columns')

Loaded 139,236 rows × 81 columns


## 2 · Initial exploration

In [3]:
print('Shape:', df.shape)
print('\nColumn dtypes:')
print(df.dtypes)
df.head(3)

Shape: (139236, 81)

Column dtypes:
Provider Code                                object
Procedure                                    object
Revision Flag                                 int64
Year                                         object
Age Band                                     object
                                             ...   
Knee Replacement Post-Op Q Confidence         int64
Knee Replacement Post-Op Q Shopping           int64
Knee Replacement Post-Op Q Stairs             int64
Knee Replacement Post-Op Q Score            float64
Knee Replacement OKS Post-Op Q Predicted    float64
Length: 81, dtype: object


,Provider Code,Procedure,Revision Flag,Year,Age Band,Gender,Pre-Op Q Assisted,Pre-Op Q Assisted By,Pre-Op Q Symptom Period,Pre-Op Q Previous Surgery,Pre-Op Q Living Arrangements,Pre-Op Q Disability,Heart Disease,High Bp,Stroke,Circulation,Lung Disease,Diabetes,Kidney Disease,Nervous System,Liver Disease,Cancer,Depression,Arthritis,Pre-Op Q Mobility,Pre-Op Q Self-Care,Pre-Op Q Activity,Pre-Op Q Discomfort,Pre-Op Q Anxiety,Pre-Op Q EQ5D Index Profile,Pre-Op Q EQ5D Index,Post-Op Q Assisted,Post-Op Q Assisted By,Post-Op Q Living Arrangements,Post-Op Q Disability,Post-Op Q Mobility,Post-Op Q Self-Care,Post-Op Q Activity,Post-Op Q Discomfort,Post-Op Q Anxiety,Post-Op Q Satisfaction,Post-Op Q Sucess,Post-Op Q Allergy,Post-Op Q Bleeding,Post-Op Q Wound,Post-Op Q Urine,Post-Op Q Further Surgery,Post-Op Q Readmitted,Post-Op Q EQ5D Index Profile,Post-Op Q EQ5D Index,Knee Replacement EQ 5D Index Post-Op Q Predicted,Pre-Op Q EQ VAS,Post-Op Q EQ VAS,Knee Replacement EQ VAS_Post-Op Q Predicted,Knee Replacement Pre-Op Q Pain,Knee Replacement Pre-Op Q Night Pain,Knee Replacement Pre-Op Q Washing,Knee Replacement Pre-Op Q Transport,Knee Replacement Pre-Op Q Walking,Knee Replacement Pre-Op Q Standing,Knee Replacement Pre-Op Q Limping,Knee Replacement Pre-Op Q Kneeling,Knee Replacement Pre-Op Q Work,Knee Replacement Pre-Op Q Confidence,Knee Replacement Pre-Op Q Shopping,Knee Replacement Pre-Op Q Stairs,Knee Replacement Pre-Op Q Score,Knee Replacement Post-Op Q Pain,Knee Replacement Post-Op Q Night Pain,Knee Replacement Post-Op Q Washing,Knee Replacement Post-Op Q Transport,Knee Replacement Post-Op Q Walking,Knee Replacement Post-Op Q Standing,Knee Replacement Post-Op Q Limping,Knee Replacement Post-Op Q Kneeling,Knee Replacement Post-Op Q Work,Knee Replacement Post-Op Q Confidence,Knee Replacement Post-Op Q Shopping,Knee Replacement Post-Op Q Stairs,Knee Replacement Post-Op Q Score,Knee Replacement OKS Post-Op Q Predicted
0,ADP02,Knee Replacement,0,2016/17,*,*,2,0,4,2,2,1,9,9,9,9,9,9,9,9,9,9,1,9,2,2,2,3,2,22232,-0.02,2,9,2,1,2,9,2,2,3,2,1,2,2,2,2,2,2,29223,NaN,NaN,63,50,59.82,0,0,2,3,2,1,0,0,1,2,2,2,15.0,3,2,3,3,2,2,3,2,3,1,3,2,29.0,28.12
1,ADP02,Knee Replacement,0,2016/17,*,*,2,0,1,2,2,2,9,9,9,9,9,9,9,9,9,9,1,9,2,2,3,2,2,22322,0.19,2,9,2,2,1,1,2,2,2,3,1,1,2,2,2,2,1,11222,0.69,0.67,30,58,64.73,3,2,4,3,4,3,1,1,1,3,2,3,30.0,4,1,3,2,4,4,4,2,4,4,4,3,39.0,39.59
2,ADP02,Knee Replacement,0,2016/17,*,*,2,0,2,2,1,2,9,9,9,9,9,9,9,9,9,9,9,1,2,1,3,3,1,21331,0.10,2,9,1,2,1,1,1,1,1,2,1,2,2,2,2,2,2,11111,1.00,0.79,65,80,77.92,0,1,4,2,2,1,2,3,1,3,2,3,24.0,3,3,4,4,4,4,4,3,4,4,4,4,45.0,40.25


## 3 · Standardise missing-value codes → `NaN`

The NHS PROMs data uses several conventions for missing / suppressed values:
- **`9`** — the dictionary-defined missing code for most categorical fields
- **`*`** — statistical suppression applied to small counts (< 5) by NHS Digital
- **`999`** — occasionally used for out-of-range / missing numeric entries
- **`''`** — empty string from CSV export

Converting all of these to `NaN` before any imputation or deletion step ensures that
downstream pandas methods (`.isna()`, `.fillna()`, `.dropna()`) behave consistently.

In [4]:
df.replace({9: np.nan, '*': np.nan, '': np.nan, 999: np.nan}, inplace=True)

total_missing = df.isna().sum().sum()
print(f'Total missing cells after standardisation: {total_missing:,}')

Total missing cells after standardisation: 1,807,003


## 4 · Comorbidity flags — treat missing as absent

These 12 columns are binary indicators (1 = condition present, 9 = not reported).  
In line with standard epidemiological practice for administrative datasets, a missing
comorbidity flag is treated as *not present* (0) rather than excluded.  
Rationale: patients with unrecorded comorbidities are unlikely to differ systematically
from patients who answered 'No'; dropping these rows would introduce selection bias.

In [5]:
COMORBIDITY_COLS = [
    'Arthritis', 'Cancer', 'Circulation', 'Depression', 'Diabetes',
    'Heart Disease', 'High Bp', 'Kidney Disease', 'Liver Disease',
    'Lung Disease', 'Nervous System', 'Stroke'
]

for col in COMORBIDITY_COLS:
    if col in df.columns:
        df[col] = df[col].fillna(0).astype(int)

print('Comorbidity prevalence (proportion of patients):')
print(df[COMORBIDITY_COLS].mean().round(3))

Comorbidity prevalence (proportion of patients):
Arthritis         7.70e-01
Cancer            5.40e-02
Circulation       5.70e-02
Depression        9.40e-02
Diabetes          1.25e-01
Heart Disease     9.40e-02
High Bp           4.42e-01
Kidney Disease    2.10e-02
Liver Disease     6.00e-03
Lung Disease      9.20e-02
Nervous System    1.00e-02
Stroke            1.70e-02
dtype: float64


## 5 · Oxford Knee Score (OKS) pre-op items — drop incomplete responders

The 12 Oxford Knee Score items are the primary disease-specific measure for knee replacement.  
Each item is mandatory for the score to be clinically valid.  
Missing rates are very low (< 1 %), so listwise deletion is preferred over imputation:
- Imputing item-level OKS responses would introduce noise into the most predictive feature group.
- The literature (Murray et al., 2007) requires a complete 12-item response for a valid OKS.  

Note: Unlike the hip dataset where `Pre-Op Q Limping` is the top predictor, for knee
the shopping and washing subscales show the highest correlation with the post-op outcome.

In [6]:
OKS_PRE_ITEMS = [
    'Knee Replacement Pre-Op Q Pain',
    'Knee Replacement Pre-Op Q Night Pain',
    'Knee Replacement Pre-Op Q Washing',
    'Knee Replacement Pre-Op Q Transport',
    'Knee Replacement Pre-Op Q Work',
    'Knee Replacement Pre-Op Q Shopping',
    'Knee Replacement Pre-Op Q Walking',
    'Knee Replacement Pre-Op Q Stairs',
    'Knee Replacement Pre-Op Q Standing',
    'Knee Replacement Pre-Op Q Confidence',
    'Knee Replacement Pre-Op Q Kneeling',
    'Knee Replacement Pre-Op Q Limping',
]

n_before = len(df)
df = df.dropna(subset=OKS_PRE_ITEMS)
print(f'Removed {n_before - len(df):,} rows with any missing OKS pre-op item.')
print(f'Remaining rows: {len(df):,}')

Removed 1,669 rows with any missing OKS pre-op item.
Remaining rows: 137,567


## 6 · Derive OKS pre-op and post-op scores from items

The Oxford Knee Score is the simple sum of its 12 items (range 0–48).  
Where the composite score is missing but all 12 items are present (data entry omission),
it is back-calculated rather than left null — preserving these rows for modelling.

In [7]:
# Pre-op OKS score: fill from items where the score cell is empty
df['Knee Replacement Pre-Op Q Score'] = df['Knee Replacement Pre-Op Q Score'].fillna(
    df[OKS_PRE_ITEMS].sum(axis=1)
)

# Post-op OKS items (same 12 dimensions, post-operative timepoint)
OKS_POST_ITEMS = [
    'Knee Replacement Post-Op Q Pain',
    'Knee Replacement Post-Op Q Night Pain',
    'Knee Replacement Post-Op Q Washing',
    'Knee Replacement Post-Op Q Transport',
    'Knee Replacement Post-Op Q Work',
    'Knee Replacement Post-Op Q Shopping',
    'Knee Replacement Post-Op Q Walking',
    'Knee Replacement Post-Op Q Stairs',
    'Knee Replacement Post-Op Q Standing',
    'Knee Replacement Post-Op Q Confidence',
    'Knee Replacement Post-Op Q Kneeling',
    'Knee Replacement Post-Op Q Limping',
]

# Only fill where all 12 post-op items exist (sum is only valid with a complete response)
post_complete = df[OKS_POST_ITEMS].notna().all(axis=1)
df.loc[post_complete, 'Knee Replacement Post-Op Q Score'] = (
    df.loc[post_complete, 'Knee Replacement Post-Op Q Score']
      .fillna(df.loc[post_complete, OKS_POST_ITEMS].sum(axis=1))
)

print('OKS pre-op score  — missing after imputation:',
      df['Knee Replacement Pre-Op Q Score'].isna().sum())
print('OKS post-op score — missing after imputation:',
      df['Knee Replacement Post-Op Q Score'].isna().sum())

OKS pre-op score  — missing after imputation: 0
OKS post-op score — missing after imputation: 2516


## 7 · EQ-5D dimensions — drop rows with missing pre-op or post-op responses

The EQ-5D (5 dimensions × 3 levels) is the generic quality-of-life instrument used
alongside the OKS. Each dimension is required to compute a valid EQ-5D utility index.

**Why listwise deletion and not imputation?**  
Valid EQ-5D levels range from 1 to 3; there is no valid '0'. Imputing a value outside
the allowed range would make the index calculation invalid. Imputing within-range values
risks masking genuine 'non-responders', which is a clinically distinct sub-group.
Missing rates are 2–4 %, so the data loss is acceptable.

In [8]:
EQ5D_DIMS = [
    'Pre-Op Q Mobility',   'Pre-Op Q Self-Care',  'Pre-Op Q Activity',
    'Pre-Op Q Discomfort', 'Pre-Op Q Anxiety',
]

n_before = len(df)
df = df.dropna(subset=EQ5D_DIMS)
print(f'Removed {n_before - len(df):,} rows with missing EQ-5D dimension responses.')
print(f'Remaining rows: {len(df):,}')

Removed 7,316 rows with missing EQ-5D dimension responses.
Remaining rows: 130,251


## 8 · Pre-Op Q Assisted — mode imputation (missing → 0 = No)

This flag records whether the patient needed help to complete the questionnaire  
(0 = No, 1 = Yes). Missing rate is ~1 %. The dominant response is 'No', and  
patients who did not record this field are indistinguishable from self-completers,  
making mode imputation safe for this administrative field.

In [9]:
df['Pre-Op Q Assisted'] = df['Pre-Op Q Assisted'].fillna(0)  # 0 = No
print('Pre-Op Q Assisted — missing after imputation:',
      df['Pre-Op Q Assisted'].isna().sum())

Pre-Op Q Assisted — missing after imputation: 0


## 9 · Pre-Op Q Symptom Period — listwise deletion

Symptom duration is a clinically meaningful predictor of post-operative outcome:  
longer pre-operative suffering is associated with worse post-op scores.  
Because this variable has strong predictive power, imputation (even mode) risks  
introducing noise into a high-signal feature. Missing rate is < 1 %; the row  
count lost is negligible.

In [10]:
n_before = len(df)
df = df.dropna(subset=['Pre-Op Q Symptom Period'])
print(f'Removed {n_before - len(df):,} rows with missing Pre-Op Q Symptom Period.')
print(f'Remaining rows: {len(df):,}')

Removed 801 rows with missing Pre-Op Q Symptom Period.
Remaining rows: 129,450


## 10 · Pre-Op Q Previous Surgery — mode imputation (missing → 0 = No)

Whether the patient had previous surgery on the same joint  
(0 = No, 1 = Yes). Missing rate is < 1 %; mode (No) imputation is appropriate  
as most patients are primary procedures.

In [11]:
df['Pre-Op Q Previous Surgery'] = df['Pre-Op Q Previous Surgery'].fillna(0)  # 0 = No
print('Pre-Op Q Previous Surgery — missing after imputation:',
      df['Pre-Op Q Previous Surgery'].isna().sum())

Pre-Op Q Previous Surgery — missing after imputation: 0


## 11 · Pre-Op Q Living Arrangements — retain 'Unknown' as a category

Living situation (1 = with family/partner, 2 = alone, 3 = nursing home, 4 = other)  
is a social support indicator. Patients who did not answer this question may represent  
a distinct social group (e.g. reluctance to disclose living alone).  
Retaining the non-response as a separate category (9 = 'Not Specified') preserves  
this potential signal rather than arbitrarily assigning them to a living group.

In [12]:
# Fill NaN with 9 to represent 'Not Specified' as an explicit category
df['Pre-Op Q Living Arrangements'] = df['Pre-Op Q Living Arrangements'].fillna(9)
print('Living Arrangements value counts:')
print(df['Pre-Op Q Living Arrangements'].value_counts())

Living Arrangements value counts:
Pre-Op Q Living Arrangements
1.0    99531
2.0    28246
9.0     1118
4.0      439
3.0      116
Name: count, dtype: int64


## 12 · Pre-Op Q Disability — retain 'Not Disclosed' as a category

Whether the patient has a long-term disability (1 = Yes, 2 = No).  
This has the highest missingness of any non-EQ-5D variable (~6 %).  
Patients may deliberately not self-identify as disabled; that choice is  
potentially informative. Treating non-response as a third level (9 = 'Not Disclosed')  
avoids introducing imputation bias on a high-missingness, sensitive question.

Also replaced 2 = No by 0 = No for consitency reasons.

In [13]:
# Fill NaN with 0 to represent 'No' as an explicit category
df['Pre-Op Q Disability'] = df['Pre-Op Q Disability'].fillna(0)
df['Pre-Op Q Disability'] = df['Pre-Op Q Disability'].replace(2, 0)
print('Disability value counts:')
print(df['Pre-Op Q Disability'].value_counts())

# Fill NaN with 0 to represent 'No' as an explicit category
df['Post-Op Q Disability'] = df['Post-Op Q Disability'].fillna(0)
df['Post-Op Q Disability'] = df['Post-Op Q Disability'].replace(2, 0)
print('Disability value counts:')
print(df['Post-Op Q Disability'].value_counts())

Disability value counts:
Pre-Op Q Disability
0.0    66998
1.0    62452
Name: count, dtype: int64
Disability value counts:
Post-Op Q Disability
0.0    87183
1.0    42267
Name: count, dtype: int64


## 13 · Derive EQ-5D Index from profile (UK value set)

The EQ-5D-3L utility index is calculated from the five-digit profile using the  
UK time trade-off value set (Dolan et al., 1997).  

This function is used to:
1. **Verify** the recorded index values against the formula (quality check).
2. **Impute** any rows where the index is missing but the five dimension values exist —
   avoiding unnecessary data loss.

In [14]:
def eq5d_index_from_profile(profile) -> float:
    """Calculate EQ-5D-3L utility index using the UK (Dolan 1997) value set.

    Parameters
    ----------
    profile : int or str
        Five-digit EQ-5D profile, e.g. 11221.  Each digit is 1, 2 or 3.

    Returns
    -------
    float or NaN if the profile is invalid.
    """
    if pd.isna(profile) or len(str(int(profile))) != 5:
        return np.nan
    try:
        digits = [int(d) for d in str(int(profile))]
    except (ValueError, TypeError):
        return np.nan
    if any(d not in (1, 2, 3) for d in digits):
        return np.nan

    # UK TTO value set constants
    CONSTANT = 0.081
    N3_PENALTY = 0.269  # applied if any dimension = 3
    WEIGHTS = [
        {'2': 0.069, '3': 0.314},  # Mobility
        {'2': 0.104, '3': 0.214},  # Self-Care
        {'2': 0.036, '3': 0.094},  # Usual Activities
        {'2': 0.123, '3': 0.386},  # Pain / Discomfort
        {'2': 0.071, '3': 0.236},  # Anxiety / Depression
    ]

    deduction = CONSTANT
    has_level_3 = False
    for i, level in enumerate(digits):
        if level == 2:
            deduction += WEIGHTS[i]['2']
        elif level == 3:
            deduction += WEIGHTS[i]['3']
            has_level_3 = True
    if has_level_3:
        deduction += N3_PENALTY

    return round(1.0 - deduction, 6)

In [15]:
PRE_EQ5D_DIMS  = ['Pre-Op Q Mobility',  'Pre-Op Q Self-Care',  'Pre-Op Q Activity',
                   'Pre-Op Q Discomfort', 'Pre-Op Q Anxiety']
POST_EQ5D_DIMS = ['Post-Op Q Mobility', 'Post-Op Q Self-Care', 'Post-Op Q Activity',
                   'Post-Op Q Discomfort','Post-Op Q Anxiety']

# Fill missing profiles by concatenating the five dimension scores
df['Pre-Op Q EQ5D Index Profile'] = df['Pre-Op Q EQ5D Index Profile'].fillna(
    df[PRE_EQ5D_DIMS].astype(int).astype(str).agg(''.join, axis=1)
)

# Fill missing index values from the (now complete) profiles
df['Pre-Op Q EQ5D Index'] = df['Pre-Op Q EQ5D Index'].fillna(
    df['Pre-Op Q EQ5D Index Profile'].apply(eq5d_index_from_profile)
)
df['Post-Op Q EQ5D Index'] = df['Post-Op Q EQ5D Index'].fillna(
    df['Post-Op Q EQ5D Index Profile'].apply(eq5d_index_from_profile)
)

print('Pre-Op EQ5D Index  — missing:', df['Pre-Op Q EQ5D Index'].isna().sum())
print('Post-Op EQ5D Index — missing:', df['Post-Op Q EQ5D Index'].isna().sum())

Pre-Op EQ5D Index  — missing: 0
Post-Op EQ5D Index — missing: 5549


## 14 · Drop post-operative detail columns (keep outcome scores only)

The 12 individual OKS post-op item responses and most post-operative questionnaire  
fields are recorded *after* surgery and are therefore leakage features — they cannot  
be known at the time of prediction. Only the composite outcome scores (OKS and EQ-5D)  
are retained as target variables.

Columns removed:
- 12 OKS post-op item responses (individual Oxford Knee Score dimensions)
- Post-op admin fields (assisted, readmitted, satisfaction, complications, etc.)
- Predicted score columns (model artefacts from NHS Digital's own regression)  
  — `Knee Replacement OKS Post-Op Q Predicted`, `Knee Replacement EQ VAS_Post-Op Q Predicted`,  
    `Knee Replacement EQ 5D Index Post-Op Q Predicted`
- `Procedure` (constant — all rows are Knee Replacement)

In [16]:
# Outcome columns to KEEP (target variables)
KEEP_POST_OP = {
    'Post-Op Q EQ5D Index Profile',
    'Post-Op Q EQ5D Index',
    'Post-Op Q EQ VAS',
    'Knee Replacement Post-Op Q Score',
}

# Drop all other post-op columns and NHS-modelled predicted columns
drop_cols = [
    col for col in df.columns
    if ('Post-Op' in col and col not in KEEP_POST_OP)
    or 'Predicted' in col
    or col in {'Procedure'}
]

df = df.drop(columns=drop_cols, errors='ignore')
print(f'Dropped {len(drop_cols)} columns. Remaining: {df.shape[1]}')
print('Kept columns:', df.columns.tolist())

Dropped 33 columns. Remaining: 48
Kept columns: ['Provider Code', 'Revision Flag', 'Year', 'Age Band', 'Gender', 'Pre-Op Q Assisted', 'Pre-Op Q Assisted By', 'Pre-Op Q Symptom Period', 'Pre-Op Q Previous Surgery', 'Pre-Op Q Living Arrangements', 'Pre-Op Q Disability', 'Heart Disease', 'High Bp', 'Stroke', 'Circulation', 'Lung Disease', 'Diabetes', 'Kidney Disease', 'Nervous System', 'Liver Disease', 'Cancer', 'Depression', 'Arthritis', 'Pre-Op Q Mobility', 'Pre-Op Q Self-Care', 'Pre-Op Q Activity', 'Pre-Op Q Discomfort', 'Pre-Op Q Anxiety', 'Pre-Op Q EQ5D Index Profile', 'Pre-Op Q EQ5D Index', 'Post-Op Q EQ5D Index Profile', 'Post-Op Q EQ5D Index', 'Pre-Op Q EQ VAS', 'Post-Op Q EQ VAS', 'Knee Replacement Pre-Op Q Pain', 'Knee Replacement Pre-Op Q Night Pain', 'Knee Replacement Pre-Op Q Washing', 'Knee Replacement Pre-Op Q Transport', 'Knee Replacement Pre-Op Q Walking', 'Knee Replacement Pre-Op Q Standing', 'Knee Replacement Pre-Op Q Limping', 'Knee Replacement Pre-Op Q Kneeling', 'Kne

## 15 · Age Band and Gender — listwise deletion

Both `Age Band` and `Gender` are suppressed together in the raw data (marked `*`)  
for patient privacy when a cell would contain < 5 observations.  
They account for ~7 % of rows. Because age and gender are key casemix-adjustment  
variables (NHS Digital uses them in all PROMs adjustment models), imputation is  
not appropriate — any imputed demographic would distort case-mix analysis.  
The data loss is acceptable given the overall sample size.

In [ ]:
n_before = len(df)
df = df.dropna(subset=['Age Band', 'Gender'])
print(f'Removed {n_before - len(df):,} rows with suppressed Age Band / Gender.')
print(f'Remaining rows: {len(df):,}')

## 16 · Encode and label categorical variables

Four encoding steps taken directly from the Gino.ipynb exploratory analysis, applied here
in the final preparation pipeline so the saved dataset is ready for modelling without
any further recoding:

1. **Binary recoding** — `Pre-Op Q Assisted` and `Pre-Op Q Previous Surgery` use `2` for "No"
   in the raw dictionary. This is recoded to `0` (standard binary: 0 = No, 1 = Yes) so these
   columns behave like all other 0/1 flags in the dataset.

2. **Gender labels** — Raw codes (`"1"`, `"2"`, `"0"`) are mapped to readable strings
   (`Male`, `Female`, `Not known`) to prevent silent errors in downstream groupby operations.

3. **Living Arrangements labels** — Numeric codes are mapped to the full descriptive strings
   from the NHS PROMs data dictionary, including the `9 = 'Not Specified'` category added in
   Step 11.

4. **Explicit numeric casting** — Columns that should be `float64` but may still be `object`
   dtype (due to mixed-type inference across the three annual CSV files) are forced to numeric.
   `errors='coerce'` converts any residual non-numeric strings to `NaN` rather than raising.

In [17]:
# ── 16a | Binary recoding ──────────────────────────────────────────────────────
# Raw data dictionary: 1 = Yes, 2 = No.  Recode 2 → 0 so all binary flags
# follow the same convention (0 = No, 1 = Yes) expected by sklearn and statsmodels.

df['Pre-Op Q Assisted']        = df['Pre-Op Q Assisted'].replace(2, 0)
df['Pre-Op Q Previous Surgery'] = df['Pre-Op Q Previous Surgery'].replace(2, 0)

print('Pre-Op Q Assisted value counts (0=No, 1=Yes):')
print(df['Pre-Op Q Assisted'].value_counts())
print('\nPre-Op Q Previous Surgery value counts (0=No, 1=Yes):')
print(df['Pre-Op Q Previous Surgery'].value_counts())

Pre-Op Q Assisted value counts (0=No, 1=Yes):
Pre-Op Q Assisted
0.0    109437
1.0     20013
Name: count, dtype: int64

Pre-Op Q Previous Surgery value counts (0=No, 1=Yes):
Pre-Op Q Previous Surgery
0.0    119534
1.0      9916
Name: count, dtype: int64


In [18]:
# ── 16b | Gender labels ────────────────────────────────────────────────────────
# Raw codes arrive as strings ("1", "2", "0") or occasionally integers after
# CSV concatenation.  Both forms are handled to avoid silent NaN introduction.
# Source: NHS PROMs data dictionary — Gender field.
''
''
''

df['Gender'] = df['Gender'].fillna(9)

GENDER_MAP = {
    '1': 'Male',   1: 'Male',
    '2': 'Female', 2: 'Female',
    'nan': 'Not known', 9: 'Not known'
}

df['Gender'] = df['Gender'].replace(GENDER_MAP)

print('Gender distribution:')
print(df['Gender'].value_counts(normalize=True).round(3))


df['Gender'].unique()

Gender distribution:
Gender
Female       0.53
Male         0.40
Not known    0.07
Name: proportion, dtype: float64


array(['Not known', 'Male', 'Female'], dtype=object)

In [19]:
# ── 16c | Living Arrangements labels ──────────────────────────────────────────
# Maps numeric codes to full descriptive strings from the NHS PROMs dictionary.
# Code 9 ('Not Specified') was introduced in Step 11 for patients who did not answer.

LIVING_MAP = {
    1: 'Living with partner/spouse/family/friends',
    2: 'Alone',
    3: 'Living in a nursing home, hospital or other long-term care home',
    4: 'Other',
    9: 'Not Specified',
}
df['Pre-Op Q Living Arrangements'] = df['Pre-Op Q Living Arrangements'].replace(LIVING_MAP)

print('Living Arrangements distribution:')
print(df['Pre-Op Q Living Arrangements'].value_counts(normalize=True).round(3))

df['Pre-Op Q Living Arrangements'].unique()

Living Arrangements distribution:
Pre-Op Q Living Arrangements
Living with partner/spouse/family/friends                          7.69e-01
Alone                                                              2.18e-01
Not Specified                                                      9.00e-03
Other                                                              3.00e-03
Living in a nursing home, hospital or other long-term care home    1.00e-03
Name: proportion, dtype: float64


array(['Alone', 'Living with partner/spouse/family/friends',
       'Not Specified', 'Other',
       'Living in a nursing home, hospital or other long-term care home'],
      dtype=object)

In [20]:
# ── 16d | Explicit numeric casting ────────────────────────────────────────────
# After concatenating three annual CSVs, pandas sometimes infers object dtype for
# columns that contain a mix of numeric values and residual string artefacts.
# pd.to_numeric(..., errors='coerce') forces float64 and converts any remaining
# non-numeric strings to NaN (which are then visible in the Step 17 summary).

NUMERIC_COLS = [
    # EQ-5D dimensions (already confirmed numeric, but cast defensively)
    'Pre-Op Q Mobility', 'Pre-Op Q Self-Care', 'Pre-Op Q Activity',
    'Pre-Op Q Discomfort', 'Pre-Op Q Anxiety',
    # EQ-5D derived scores
    'Pre-Op Q EQ5D Index', 'Post-Op Q EQ5D Index',
    'Pre-Op Q EQ VAS',     'Post-Op Q EQ VAS',
    # Administrative / casemix fields
    'Pre-Op Q Assisted', 'Pre-Op Q Assisted By',
    'Pre-Op Q Symptom Period', 'Pre-Op Q Previous Surgery',
    'Pre-Op Q Disability', 'Revision Flag',
    # Oxford Knee Score — 12 pre-op items + composite
    'Knee Replacement Pre-Op Q Pain',        'Knee Replacement Pre-Op Q Night Pain',
    'Knee Replacement Pre-Op Q Washing',     'Knee Replacement Pre-Op Q Transport',
    'Knee Replacement Pre-Op Q Work',        'Knee Replacement Pre-Op Q Shopping',
    'Knee Replacement Pre-Op Q Walking',     'Knee Replacement Pre-Op Q Stairs',
    'Knee Replacement Pre-Op Q Standing',    'Knee Replacement Pre-Op Q Confidence',
    'Knee Replacement Pre-Op Q Kneeling',    'Knee Replacement Pre-Op Q Limping',
    'Knee Replacement Pre-Op Q Score',
    # Outcome (target variable)
    'Knee Replacement Post-Op Q Score',
]

for col in NUMERIC_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print('Numeric casting complete.')
print('\nDtype check for score columns:')
print(df[[c for c in NUMERIC_COLS if c in df.columns]].dtypes)

Numeric casting complete.

Dtype check for score columns:
Pre-Op Q Mobility                       float64
Pre-Op Q Self-Care                      float64
Pre-Op Q Activity                       float64
Pre-Op Q Discomfort                     float64
Pre-Op Q Anxiety                        float64
Pre-Op Q EQ5D Index                     float64
Post-Op Q EQ5D Index                    float64
Pre-Op Q EQ VAS                         float64
Post-Op Q EQ VAS                        float64
Pre-Op Q Assisted                       float64
Pre-Op Q Assisted By                      int64
Pre-Op Q Symptom Period                 float64
Pre-Op Q Previous Surgery               float64
Pre-Op Q Disability                     float64
Revision Flag                             int64
Knee Replacement Pre-Op Q Pain          float64
Knee Replacement Pre-Op Q Night Pain    float64
Knee Replacement Pre-Op Q Washing       float64
Knee Replacement Pre-Op Q Transport     float64
Knee Replacement Pre-Op Q Work

## 17 · Final data summary

In [21]:
print('=== Final dataset ===')
print(f'Rows:    {len(df):,}')
print(f'Columns: {df.shape[1]}')
print(f'\nRemaining missing values:')
missing = df.isna().sum()
print(missing[missing > 0])
df.describe(include='all')

=== Final dataset ===
Rows:    129,450
Columns: 48

Remaining missing values:
Age Band                            8786
Post-Op Q EQ5D Index                5549
Pre-Op Q EQ VAS                     8179
Post-Op Q EQ VAS                    5890
Knee Replacement Post-Op Q Score    2306
dtype: int64


,Provider Code,Revision Flag,Year,Age Band,Gender,Pre-Op Q Assisted,Pre-Op Q Assisted By,Pre-Op Q Symptom Period,Pre-Op Q Previous Surgery,Pre-Op Q Living Arrangements,Pre-Op Q Disability,Heart Disease,High Bp,Stroke,Circulation,Lung Disease,Diabetes,Kidney Disease,Nervous System,Liver Disease,Cancer,Depression,Arthritis,Pre-Op Q Mobility,Pre-Op Q Self-Care,Pre-Op Q Activity,Pre-Op Q Discomfort,Pre-Op Q Anxiety,Pre-Op Q EQ5D Index Profile,Pre-Op Q EQ5D Index,Post-Op Q EQ5D Index Profile,Post-Op Q EQ5D Index,Pre-Op Q EQ VAS,Post-Op Q EQ VAS,Knee Replacement Pre-Op Q Pain,Knee Replacement Pre-Op Q Night Pain,Knee Replacement Pre-Op Q Washing,Knee Replacement Pre-Op Q Transport,Knee Replacement Pre-Op Q Walking,Knee Replacement Pre-Op Q Standing,Knee Replacement Pre-Op Q Limping,Knee Replacement Pre-Op Q Kneeling,Knee Replacement Pre-Op Q Work,Knee Replacement Pre-Op Q Confidence,Knee Replacement Pre-Op Q Shopping,Knee Replacement Pre-Op Q Stairs,Knee Replacement Pre-Op Q Score,Knee Replacement Post-Op Q Score
count,129450,129450.00,129450,120664,129450,129450.00,129450.0,129450.00,129450.00,129450,129450.00,129450.00,129450.00,129450.00,129450.00,129450.00,129450.00,129450.00,129450.00,1.29e+05,129450.00,129450.00,129450.00,129450.00,129450.00,129450.00,129450.00,129450.00,129450.00,129450.00,129450.00,123901.00,121271.00,123560.00,129450.00,129450.00,129450.00,129450.00,129450.00,129450.00,129450.00,129450.00,129450.00,129450.00,129450.00,129450.00,129450.00,127144.00
unique,294,NaN,3,6,3,NaN,NaN,NaN,NaN,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,RL1,NaN,2016/17,70 to 79,Female,NaN,NaN,NaN,NaN,Living with partner/spouse/family/friends,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,2865,NaN,44199,50858,68626,NaN,NaN,NaN,NaN,99531,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,0.04,NaN,NaN,NaN,0.15,0.0,2.60,0.08,NaN,0.48,0.09,0.44,0.02,0.06,0.09,0.12,0.02,0.01,5.69e-03,0.05,0.09,0.77,1.92,1.31,2.03,2.37,1.40,20761.30,0.41,17500.29,0.75,67.34,74.84,0.56,1.23,2.82,2.07,2.05,1.68,0.89,0.82,1.42,1.84,1.85,1.78,19.00,35.93
std,NaN,0.19,NaN,NaN,NaN,0.36,0.0,0.87,0.27,NaN,0.50,0.29,0.50,0.13,0.23,0.29,0.33,0.14,0.10,7.52e-02,0.23,0.29,0.42,0.28,0.48,0.47,0.50,0.57,2916.77,0.31,12706.46,0.25,20.59,18.27,0.64,1.15,1.00,0.83,1.10,0.83,1.01,0.86,0.87,1.18,1.21,0.89,7.76,9.46
min,NaN,0.00,NaN,NaN,NaN,0.00,0.0,1.00,0.00,NaN,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00e+00,0.00,0.00,0.00,1.00,1.00,1.00,1.00,1.00,11111.00,-0.59,11111.00,-0.59,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,NaN,0.00,NaN,NaN,NaN,0.00,0.0,2.00,0.00,NaN,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00e+00,0.00,0.00,1.00,2.00,1.00,2.00,2.00,1.00,21221.00,0.09,11111.00,0.69,50.00,65.00,0.00,0.00,2.00,2.00,2.00,1.00,0.00,0.00,1.00,1.00,1.00,1.00,13.00,30.00
50%,NaN,0.00,NaN,NaN,NaN,0.00,0.0,2.00,0.00,NaN,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00e+00,0.00,0.00,1.00,2.00,1.00,2.00,2.00,1.00,21231.00,0.59,11221.00,0.76,70.00,80.00,0.00,1.00,3.00,2.00,2.00,2.00,1.00,1.00,1.00,2.00,2.00,2.00,19.00,38.00
75%,NaN,0.00,NaN,NaN,NaN,0.00,0.0,3.00,0.00,NaN,1.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00e+00,0.00,0.00,1.00,2.00,2.00,2.00,3.00,2.00,22221.00,0.69,21221.00,1.00,80.00,90.00,1.00,2.00,4.00,2.00,3.00,2.00,1.00,1.00,2.00,3.00,3.00,2.00,24.00,43.00


## 18 · Save prepared dataset

The cleaned dataset is saved as **Parquet** (columnar format) to  
`data/interim/knee-provider.parquet`.

Parquet is preferred over CSV for interim data because:
- dtypes are preserved exactly (no re-parsing needed)
- file size is significantly smaller due to columnar compression
- read/write speeds are faster for downstream notebooks

In [23]:
from pathlib import Path
import pandas as pd

# Save to a 'data' folder in the notebook's parent directory
OUTPUT = Path.cwd().parent / 'data' / 'Knee-provider.parquet'

# Convert Period columns to string for Parquet compatibility
# Fixed: is_period_dtype() removed in pandas 2.x
for col in df.columns:
    if isinstance(df[col].dtype, pd.PeriodDtype):
        df[col] = df[col].astype(str)

# Convert object columns to string for Parquet compatibility
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype('string')

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUTPUT, index=False, engine='pyarrow')
print(f'Saved prepared dataset to: {OUTPUT.resolve()}')
print(f'File size: {OUTPUT.stat().st_size / 1024:.1f} KB')

Saved prepared dataset to: C:\Users\mittall\source\EAISI\NHS\EAISI-NHS\notebooks\Final\data\Knee-provider.parquet
File size: 1953.3 KB
